Alasan dilakukan eksperimen max lenght

"Karena komentar pada dataset memiliki panjang yang bervariasi dan sebagian relatif panjang, dilakukan eksperimen maximum sequence length untuk mengetahui panjang input yang memberikan performa klasifikasi yang optimal tanpa kehilangan konteks secara berlebihan."

CPU: AMD Ryzen 7 5800H

Core/thread: 8 core / 16 logical processors

RAM: 15,4 GB usable → praktis 16 GB RAM

Storage: SSD NVMe

GPU: AMD Radeon(TM) Graphics


Environment training

Google Colab + NVIDIA Tesla T4

In [ ]:
# ============================================================
# CELL 1 — CHECK ENVIRONMENT
# ============================================================

import sys
import os
import platform
import torch
import transformers

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

print(f"Python       : {sys.version}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"Platform     : {platform.platform()}")

print("\nCUDA")
print(f"CUDA available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version   : {torch.version.cuda}")
    print(
        f"GPU memory     : "
        f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
    )
else:
    print("WARNING: CUDA/GPU tidak tersedia.")

print("=" * 60)

ENVIRONMENT CHECK
Python       : 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch      : 2.11.0+cu128
Transformers : 5.15.1
Platform     : Linux-6.6.122+-x86_64-with-glibc2.35

CUDA
CUDA available : True
GPU            : Tesla T4
CUDA version   : 12.8
GPU memory     : 14.56 GB


In [ ]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES
# ============================================================

import os
import gc
import time
import random
import warnings

import numpy as np
import pandas as pd

import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


In [ ]:
# ============================================================
# CELL 3 — PROJECT STORAGE
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08"

DATA_DIR = os.path.join(PROJECT_DIR, "dataset")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
LOG_DIR = os.path.join(PROJECT_DIR, "logs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("=" * 60)
print("STORAGE SETUP")
print("=" * 60)

print(f"Project directory : {PROJECT_DIR}")
print(f"Dataset directory : {DATA_DIR}")
print(f"Result directory  : {RESULT_DIR}")
print(f"Log directory     : {LOG_DIR}")

print("\nFolder berhasil disiapkan.")

STORAGE SETUP
Project directory : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08
Dataset directory : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset
Result directory  : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/results
Log directory     : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/logs

Folder berhasil disiapkan.


In [ ]:
# ============================================================
# CELL 4 — INSPEKSI FOLDER PROJECT
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08"

print("=" * 60)
print("PROJECT DIRECTORY CHECK")
print("=" * 60)

print(f"Path : {PROJECT_DIR}")

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(
        f"Folder tidak ditemukan:\n{PROJECT_DIR}"
    )

print("\nIsi folder:\n")

for item in sorted(os.listdir(PROJECT_DIR)):
    full_path = os.path.join(PROJECT_DIR, item)

    if os.path.isdir(full_path):
        print(f"[FOLDER] {item}")
    else:
        size_mb = os.path.getsize(full_path) / (1024 ** 2)
        print(f"[FILE]   {item} ({size_mb:.2f} MB)")

print("\n" + "=" * 60)
print("PROJECT DIRECTORY TERBACA")
print("=" * 60)

PROJECT DIRECTORY CHECK
Path : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08

Isi folder:

[FILE]   Dokumen tanpa judul.gdoc (0.00 MB)
[FILE]   Fine tunning - new data preprocesed.ipynb (0.01 MB)
[FOLDER] dataset
[FOLDER] logs
[FOLDER] results

PROJECT DIRECTORY TERBACA


In [ ]:
# ============================================================
# CELL 5 — INSPEKSI FOLDER DATASET
# ============================================================

DATA_DIR = os.path.join(
    PROJECT_DIR,
    "dataset"
)

print("=" * 60)
print("DATASET DIRECTORY CHECK")
print("=" * 60)

print(f"Path : {DATA_DIR}")

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(
        f"Folder dataset tidak ditemukan:\n{DATA_DIR}"
    )

print("\nIsi folder dataset:\n")

for item in sorted(os.listdir(DATA_DIR)):
    full_path = os.path.join(DATA_DIR, item)

    if os.path.isdir(full_path):
        print(f"[FOLDER] {item}")
    else:
        size_mb = os.path.getsize(full_path) / (1024 ** 2)
        print(f"[FILE]   {item} ({size_mb:.2f} MB)")

print("\n" + "=" * 60)
print("DATASET DIRECTORY TERBACA")
print("=" * 60)

DATASET DIRECTORY CHECK
Path : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset

Isi folder dataset:

[FILE]   dataset_preprocessed_C0-C4_revised.csv (3.15 MB)

DATASET DIRECTORY TERBACA


In [ ]:
# ============================================================
# CELL 6 — LOAD DATASET REVISI & INSPEKSI AWAL
# ============================================================

DATA_PATH = os.path.join(
    DATA_DIR,
    "dataset_preprocessed_C0-C4_revised.csv"
)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset tidak ditemukan:\n{DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print(f"File          : {DATA_PATH}")
print(f"Jumlah baris  : {len(df):,}")
print(f"Jumlah kolom  : {len(df.columns)}")

print("\nNama kolom:")
print(df.columns.tolist())

print("\n5 baris pertama:")
display(df.head())

print("\nInformasi data:")
df.info()

DATASET LOADED
File          : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset/dataset_preprocessed_C0-C4_revised.csv
Jumlah baris  : 3,815
Jumlah kolom  : 13

Nama kolom:
['No', 'Komentar', 'Label Awal', 'Label Pakar', 'Pasal Dominan', 'Catatan', 'Unnamed: 6', 'Kode Label', 'jumlah_kata', 'jumlah_karakter', 'clean_comment', 'Komentar_clean', 'Komentar_preprocessed']

5 baris pertama:


,No,Komentar,Label Awal,Label Pakar,Pasal Dominan,Catatan,Unnamed: 6,Kode Label,jumlah_kata,jumlah_karakter,clean_comment,Komentar_clean,Komentar_preprocessed
0,5708.0,Apakah seorang maling anak'y jg akan menjadi m...,Indikasi penghinaan atau kebencian terhadap in...,Ambigu atau konteks tidak cukup,NaN,Pertanyaan umum tanpa target yang jelas.,NaN,C4,8.0,56.0,apakah seorang maling anak'y jg akan menjadi m...,apakah seorang maling anak'y jg akan menjadi m...,apakah seorang maling anaknya juga akan menjad...
1,495.0,"​@HanaFanda-wx1cciyaa betul,\nMeni hese ngesah...",Netral atau tidak terindikasi,Konten ofensif,NaN,Tidak terindikasi Pasal 27A maupun Pasal 28 ay...,NaN,C2,12.0,91.0,"​@hanafanda-wx1cciyaa betul, meni hese ngesahk...","​-wx1cciyaa betul, meni hese ngesahkeun pearam...","​-wx1cciyaa betul, meni hese ngesahkeun pearam..."
2,1841.0,Omongan nya pak bahlil sering kontradiksi. Lah...,Netral atau tidak terindikasi,Netral atau tidak terindikasi,NaN,NaN,NaN,C0,7.0,50.0,omongan nya pak bahlil sering kontradiksi. lah...,omongan nya pak bahlil sering kontradiksi. lah...,omongan nya pak bahlil sering kontradiksi. lah...
3,6483.0,Astagfirullah. Katanya Wakil Rakyat tapi kelak...,Indikasi penghinaan atau kebencian terhadap in...,Konten ofensif,NaN,Menuduh wakil rakyat sebagai maling dan zalim.,NaN,C2,13.0,96.0,astagfirullah. katanya wakil rakyat tapi kelak...,astagfirullah. katanya wakil rakyat tapi kelak...,astagfirullah. katanya wakil rakyat tetapi kel...
4,2222.0,"Cobalah pampang ke45 poin itu di media sosial,...",Netral atau tidak terindikasi,Ambigu atau konteks tidak cukup,NaN,komentar ini belum menunjukkan unsur yang cuku...,NaN,C4,12.0,76.0,"cobalah pampang ke45 poin itu di media sosial,...","cobalah pampang ke45 poin itu di media sosial,...","cobalah pampang ke45 poin itu di media sosial,..."



Informasi data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3815 entries, 0 to 3814
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   No                     3597 non-null   float64
 1   Komentar               3815 non-null   object 
 2   Label Awal             3815 non-null   object 
 3   Label Pakar            3815 non-null   object 
 4   Pasal Dominan          637 non-null    object 
 5   Catatan                3387 non-null   object 
 6   Unnamed: 6             1 non-null      object 
 7   Kode Label             3815 non-null   object 
 8   jumlah_kata            3597 non-null   float64
 9   jumlah_karakter        3597 non-null   float64
 10  clean_comment          3815 non-null   object 
 11  Komentar_clean         3815 non-null   object 
 12  Komentar_preprocessed  3815 non-null   object 
dtypes: float64(3), object(10)
memory usage: 387.6+ KB


In [ ]:
# ============================================================
# CELL 7 — CHECK FINAL LABELS
# ============================================================

TEXT_COL = "Komentar_preprocessed"
LABEL_COL = "Kode Label"

print("=" * 60)
print("FINAL INPUT & LABEL CHECK")
print("=" * 60)

print(f"Text column  : {TEXT_COL}")
print(f"Label column : {LABEL_COL}")

print("\nMissing value:")
print(f"{TEXT_COL} : {df[TEXT_COL].isna().sum()}")
print(f"{LABEL_COL}: {df[LABEL_COL].isna().sum()}")

print("\nLabel distribution:")
print(df[LABEL_COL].value_counts().sort_index())

print("\nUnique labels:")
print(sorted(df[LABEL_COL].unique()))

print("\nSample input + label:")
display(
    df[[TEXT_COL, LABEL_COL]].head(10)
)

FINAL INPUT & LABEL CHECK
Text column  : Komentar_preprocessed
Label column : Kode Label

Missing value:
Komentar_preprocessed : 0
Kode Label: 0

Label distribution:
Kode Label
C0    763
C1    763
C2    763
C3    763
C4    763
Name: count, dtype: int64

Unique labels:
['C0', 'C1', 'C2', 'C3', 'C4']

Sample input + label:


,Komentar_preprocessed,Kode Label
0,apakah seorang maling anaknya juga akan menjad...,C4
1,"​-wx1cciyaa betul, meni hese ngesahkeun pearam...",C2
2,omongan nya pak bahlil sering kontradiksi. lah...,C0
3,astagfirullah. katanya wakil rakyat tetapi kel...,C2
4,"cobalah pampang ke45 poin itu di media sosial,...",C4
5,hentikan mbg..motong honor rt,C4
6,namanya juga sudah diwakilkan kesejahteraannya,C0
7,"iya ih, dia tuh sehat tidak ya kejiwaannya¿¿ o...",C2
8,astaga! 23 warga koja keracunan usai santap na...,C1
9,pemerintah kok dungu.cerdas pk roky gerung..,C4


In [ ]:
# ============================================================
# CELL 8 — LABEL ENCODING & TRAIN/VALIDATION/TEST SPLIT
# ============================================================

SEED = 42

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# ============================================================
# LABEL ENCODING
# ============================================================

label2id = {
    "C0": 0,
    "C1": 1,
    "C2": 2,
    "C3": 3,
    "C4": 4
}

id2label = {
    0: "C0",
    1: "C1",
    2: "C2",
    3: "C3",
    4: "C4"
}

df["label"] = df[LABEL_COL].map(label2id)

# Pastikan tidak ada label yang gagal dipetakan
if df["label"].isna().any():
    raise ValueError("Ada label yang gagal dipetakan ke label numerik.")

df["label"] = df["label"].astype(int)

print("=" * 60)
print("LABEL ENCODING")
print("=" * 60)

print("Mapping:")
for label, idx in label2id.items():
    print(f"{label} -> {idx}")

print("\nEncoded label distribution:")
print(df["label"].value_counts().sort_index())

# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

# Reset index agar bersih
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\n" + "=" * 60)
print("DATA SPLIT")
print("=" * 60)

print(f"Train      : {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation : {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test       : {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")
print(f"Total      : {len(train_df) + len(val_df) + len(test_df):,}")

# ============================================================
# CEK DISTRIBUSI LABEL PADA SETIAP SPLIT
# ============================================================

print("\nDistribusi label TRAIN:")
print(train_df["label"].value_counts().sort_index())

print("\nDistribusi label VALIDATION:")
print(val_df["label"].value_counts().sort_index())

print("\nDistribusi label TEST:")
print(test_df["label"].value_counts().sort_index())

LABEL ENCODING
Mapping:
C0 -> 0
C1 -> 1
C2 -> 2
C3 -> 3
C4 -> 4

Encoded label distribution:
label
0    763
1    763
2    763
3    763
4    763
Name: count, dtype: int64

DATA SPLIT
Train      : 3,052 (80.0%)
Validation : 381 (10.0%)
Test       : 382 (10.0%)
Total      : 3,815

Distribusi label TRAIN:
label
0    610
1    611
2    610
3    610
4    611
Name: count, dtype: int64

Distribusi label VALIDATION:
label
0    76
1    76
2    77
3    76
4    76
Name: count, dtype: int64

Distribusi label TEST:
label
0    77
1    76
2    76
3    77
4    76
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 9 — SAVE FIXED TRAIN / VALIDATION / TEST SPLIT
# ============================================================

SPLIT_DIR = os.path.join(PROJECT_DIR, "dataset", "splits")

os.makedirs(SPLIT_DIR, exist_ok=True)

# Kolom yang disimpan untuk eksperimen fine-tuning
SAVE_COLS = [
    "Komentar_preprocessed",
    "Kode Label",
    "label"
]

train_path = os.path.join(SPLIT_DIR, "train_split.csv")
val_path = os.path.join(SPLIT_DIR, "validation_split.csv")
test_path = os.path.join(SPLIT_DIR, "test_split.csv")

train_df[SAVE_COLS].to_csv(train_path, index=False)
val_df[SAVE_COLS].to_csv(val_path, index=False)
test_df[SAVE_COLS].to_csv(test_path, index=False)

print("=" * 60)
print("FIXED SPLIT SAVED")
print("=" * 60)

print(f"Train      : {train_path}")
print(f"Validation : {val_path}")
print(f"Test       : {test_path}")

print("\nJumlah data:")
print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Test       : {len(test_df):,}")

print("\nSplit sudah dikunci dengan SEED =", SEED)

FIXED SPLIT SAVED
Train      : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset/splits/train_split.csv
Validation : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset/splits/validation_split.csv
Test       : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset/splits/test_split.csv

Jumlah data:
Train      : 3,052
Validation : 381
Test       : 382

Split sudah dikunci dengan SEED = 42


In [ ]:
# ============================================================
# CELL 10 — TOKEN LENGTH ANALYSIS
# ============================================================

ANALYSIS_MODEL = "indobenchmark/indobert-base-p1"

print("=" * 60)
print("LOADING TOKENIZER FOR TOKEN LENGTH ANALYSIS")
print("=" * 60)

print(f"Checkpoint : {ANALYSIS_MODEL}")

analysis_tokenizer = AutoTokenizer.from_pretrained(
    ANALYSIS_MODEL,
    use_fast=True
)

print(f"Tokenizer  : {type(analysis_tokenizer).__name__}")
print("Tokenizer berhasil dimuat.")

LOADING TOKENIZER FOR TOKEN LENGTH ANALYSIS
Checkpoint : indobenchmark/indobert-base-p1


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer  : BertTokenizer
Tokenizer berhasil dimuat.


In [ ]:
# ============================================================
# CELL 11 — TOKEN LENGTH DISTRIBUTION
# ============================================================

print("=" * 60)
print("TOKEN LENGTH ANALYSIS")
print("=" * 60)

# Ambil teks training
train_texts = train_df[TEXT_COL].astype(str).tolist()

# Tokenisasi tanpa truncation dan tanpa padding
tokenized_lengths = []

for text in train_texts:
    encoded = analysis_tokenizer(
        text,
        truncation=False,
        padding=False,
        add_special_tokens=True
    )

    tokenized_lengths.append(len(encoded["input_ids"]))

# Simpan hasil panjang token
train_token_lengths = np.array(tokenized_lengths)

print(f"Jumlah teks dianalisis : {len(train_token_lengths):,}")

print("\nStatistik panjang token:")
print(f"Minimum   : {train_token_lengths.min()}")
print(f"Q1 (25%)  : {np.percentile(train_token_lengths, 25):.0f}")
print(f"Median    : {np.median(train_token_lengths):.0f}")
print(f"Q3 (75%)  : {np.percentile(train_token_lengths, 75):.0f}")
print(f"Mean      : {train_token_lengths.mean():.2f}")
print(f"Maximum   : {train_token_lengths.max()}")

print("\nPersentil:")
for p in [90, 95, 97, 98, 99, 99.5]:
    print(
        f"P{p:<4} : "
        f"{np.percentile(train_token_lengths, p):.0f} token"
    )

TOKEN LENGTH ANALYSIS
Jumlah teks dianalisis : 3,052

Statistik panjang token:
Minimum   : 3
Q1 (25%)  : 12
Median    : 22
Q3 (75%)  : 39
Mean      : 34.00
Maximum   : 643

Persentil:
P90   : 68 token
P95   : 92 token
P97   : 116 token
P98   : 134 token
P99   : 221 token
P99.5 : 390 token


In [ ]:
# ============================================================
# CELL 12 — TRUNCATION ANALYSIS
# ============================================================

print("=" * 60)
print("TRUNCATION ANALYSIS")
print("=" * 60)

MAX_LENGTH_CANDIDATES = [128, 256, 512]

total_samples = len(train_token_lengths)

truncation_results = []

for max_len in MAX_LENGTH_CANDIDATES:
    truncated_count = np.sum(train_token_lengths > max_len)
    truncated_percentage = (
        truncated_count / total_samples
    ) * 100

    retained_count = total_samples - truncated_count
    retained_percentage = (
        retained_count / total_samples
    ) * 100

    truncation_results.append({
        "MAX_LENGTH": max_len,
        "Truncated_Count": int(truncated_count),
        "Truncated_Percentage": truncated_percentage,
        "Retained_Count": int(retained_count),
        "Retained_Percentage": retained_percentage
    })

truncation_df = pd.DataFrame(truncation_results)

display(
    truncation_df.style.format({
        "Truncated_Percentage": "{:.2f}%",
        "Retained_Percentage": "{:.2f}%"
    })
)

TRUNCATION ANALYSIS


,MAX_LENGTH,Truncated_Count,Truncated_Percentage,Retained_Count,Retained_Percentage
0,128,73,2.39%,2979,97.61%
1,256,26,0.85%,3026,99.15%
2,512,8,0.26%,3044,99.74%


In [ ]:
# ============================================================
# CELL 13 — CREATE HUGGING FACE DATASETS
# ============================================================

from datasets import Dataset

print("=" * 60)
print("CREATING HUGGING FACE DATASETS")
print("=" * 60)

# Gunakan hanya kolom yang dibutuhkan model
train_hf = Dataset.from_pandas(
    train_df[[TEXT_COL, "label"]],
    preserve_index=False
)

val_hf = Dataset.from_pandas(
    val_df[[TEXT_COL, "label"]],
    preserve_index=False
)

test_hf = Dataset.from_pandas(
    test_df[[TEXT_COL, "label"]],
    preserve_index=False
)

print(f"Train      : {len(train_hf):,}")
print(f"Validation : {len(val_hf):,}")
print(f"Test       : {len(test_hf):,}")

print("\nContoh train:")
print(train_hf[0])

print("\nDataset Hugging Face berhasil dibuat.")

CREATING HUGGING FACE DATASETS
Train      : 3,052
Validation : 381
Test       : 382

Contoh train:
{'Komentar_preprocessed': 'wah sudah parah kesal ini roman2 nya bang fery sampai gemeteran cuy bicara nya', 'label': 0}

Dataset Hugging Face berhasil dibuat.


In [ ]:
# ============================================================
# CELL 14 — EXPERIMENT CONFIGURATION
# 6 IndoBERT × 3 MAX_LENGTH = 18 EXPERIMENTS
# ============================================================

MODEL_CONFIGS = {
    "IndoBERT Lite P1": {
        "model_name": "indobenchmark/indobert-lite-base-p1",
        "batch_size": 16,
    },
    "IndoBERT Lite P2": {
        "model_name": "indobenchmark/indobert-lite-base-p2",
        "batch_size": 16,
    },
    "IndoBERT Base P1": {
        "model_name": "indobenchmark/indobert-base-p1",
        "batch_size": 16,
    },
    "IndoBERT Base P2": {
        "model_name": "indobenchmark/indobert-base-p2",
        "batch_size": 16,
    },
    "IndoBERT Large P1": {
        "model_name": "indobenchmark/indobert-large-p1",
        "batch_size": 4,
    },
    "IndoBERT Large P2": {
        "model_name": "indobenchmark/indobert-large-p2",
        "batch_size": 4,
    },
}

# Kandidat MAX_LENGTH berdasarkan hasil analisis token
MAX_LENGTH_CANDIDATES = [128, 256, 512]

NUM_LABELS = 5

print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)

print("\nMODEL CONFIGURATION")
print("-" * 60)

for model_label, config in MODEL_CONFIGS.items():
    print(f"{model_label}")
    print(f"  Checkpoint : {config['model_name']}")
    print(f"  Batch size : {config['batch_size']}")
    print()

print("MAX_LENGTH CANDIDATES")
print("-" * 60)
print(MAX_LENGTH_CANDIDATES)

total_experiments = (
    len(MODEL_CONFIGS) * len(MAX_LENGTH_CANDIDATES)
)

print("\n" + "=" * 60)
print(f"TOTAL EXPERIMENTS : {total_experiments}")
print("=" * 60)

EXPERIMENT CONFIGURATION

MODEL CONFIGURATION
------------------------------------------------------------
IndoBERT Lite P1
  Checkpoint : indobenchmark/indobert-lite-base-p1
  Batch size : 16

IndoBERT Lite P2
  Checkpoint : indobenchmark/indobert-lite-base-p2
  Batch size : 16

IndoBERT Base P1
  Checkpoint : indobenchmark/indobert-base-p1
  Batch size : 16

IndoBERT Base P2
  Checkpoint : indobenchmark/indobert-base-p2
  Batch size : 16

IndoBERT Large P1
  Checkpoint : indobenchmark/indobert-large-p1
  Batch size : 4

IndoBERT Large P2
  Checkpoint : indobenchmark/indobert-large-p2
  Batch size : 4

MAX_LENGTH CANDIDATES
------------------------------------------------------------
[128, 256, 512]

TOTAL EXPERIMENTS : 18


In [ ]:
# ============================================================
# CELL 15 — TOKENIZATION FUNCTION
# ============================================================

def tokenize_datasets(
    train_dataset,
    validation_dataset,
    test_dataset,
    tokenizer,
    max_length
):
    """
    Tokenisasi train, validation, dan test menggunakan
    tokenizer dari checkpoint model yang sedang diuji.

    Parameters
    ----------
    train_dataset : Hugging Face Dataset
    validation_dataset : Hugging Face Dataset
    test_dataset : Hugging Face Dataset
    tokenizer : AutoTokenizer
        Tokenizer yang sesuai dengan checkpoint model.
    max_length : int
        Kandidat maximum sequence length.

    Returns
    -------
    tokenized_train : Hugging Face Dataset
    tokenized_validation : Hugging Face Dataset
    tokenized_test : Hugging Face Dataset
    """

    def tokenize_function(examples):
        return tokenizer(
            examples[TEXT_COL],
            truncation=True,
            max_length=max_length,
            padding=False
        )

    tokenized_train = train_dataset.map(
        tokenize_function,
        batched=True,
        desc=f"Tokenizing train | max_length={max_length}"
    )

    tokenized_validation = validation_dataset.map(
        tokenize_function,
        batched=True,
        desc=f"Tokenizing validation | max_length={max_length}"
    )

    tokenized_test = test_dataset.map(
        tokenize_function,
        batched=True,
        desc=f"Tokenizing test | max_length={max_length}"
    )

    # Kolom teks tidak lagi diperlukan setelah tokenisasi.
    tokenized_train = tokenized_train.remove_columns(
        [TEXT_COL]
    )

    tokenized_validation = tokenized_validation.remove_columns(
        [TEXT_COL]
    )

    tokenized_test = tokenized_test.remove_columns(
        [TEXT_COL]
    )

    return (
        tokenized_train,
        tokenized_validation,
        tokenized_test
    )


print("=" * 60)
print("TOKENIZATION FUNCTION READY")
print("=" * 60)
print("Function : tokenize_datasets()")
print("MAX_LENGTH candidates :", MAX_LENGTH_CANDIDATES)
print("Status : READY")

TOKENIZATION FUNCTION READY
Function : tokenize_datasets()
MAX_LENGTH candidates : [128, 256, 512]
Status : READY


In [ ]:
# ============================================================
# CELL 16 — DATA COLLATOR
# ============================================================

from transformers import DataCollatorWithPadding

def create_data_collator(tokenizer):
    """
    Membuat data collator dengan dynamic padding
    berdasarkan tokenizer model yang sedang digunakan.
    """

    return DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding=True
    )


print("=" * 60)
print("DATA COLLATOR FUNCTION READY")
print("=" * 60)
print("Collator : DataCollatorWithPadding")
print("Padding  : Dynamic padding per batch")
print("Status   : READY")

DATA COLLATOR FUNCTION READY
Collator : DataCollatorWithPadding
Padding  : Dynamic padding per batch
Status   : READY


In [ ]:
# ============================================================
# CELL 17 — EVALUATION METRICS
# ============================================================

def compute_metrics(eval_pred):
    """
    Menghitung metrik evaluasi untuk klasifikasi C0-C4.

    Macro-F1 digunakan sebagai metrik utama karena
    setiap kelas perlu diperlakukan secara setara.
    """

    logits, labels = eval_pred

    # Prediksi kelas berdasarkan nilai logit tertinggi
    predictions = np.argmax(logits, axis=-1)

    # Precision, Recall, F1 per kelas lalu dirata-ratakan secara macro
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_macro": f1
    }


print("=" * 60)
print("EVALUATION METRICS READY")
print("=" * 60)

print("Metrics:")
print("- Accuracy")
print("- Macro Precision")
print("- Macro Recall")
print("- Macro F1")

print("\nPrimary metric : Macro-F1")
print("Status         : READY")

EVALUATION METRICS READY
Metrics:
- Accuracy
- Macro Precision
- Macro Recall
- Macro F1

Primary metric : Macro-F1
Status         : READY


In [ ]:
# ============================================================
# CELL 18 — TRAINING ARGUMENTS
# EARLY STOPPING
# ============================================================

from transformers import EarlyStoppingCallback


def create_training_arguments(
    experiment_name,
    batch_size,
    output_dir="/content/indobert_tmp"
):
    """
    Membuat TrainingArguments untuk satu eksperimen.

    num_train_epochs digunakan sebagai batas maksimum training,
    sedangkan penghentian aktual ditentukan oleh Early Stopping.
    """

    experiment_output_dir = os.path.join(
        output_dir,
        experiment_name.replace(" ", "_")
    )

    training_args = TrainingArguments(
        output_dir=experiment_output_dir,

        # ====================================================
        # TRAINING
        # ====================================================

        # Batas maksimum epoch.
        # Training dapat berhenti lebih awal melalui
        # EarlyStoppingCallback.
        num_train_epochs=10,

        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,

        learning_rate=2e-5,
        weight_decay=0.01,

        # ====================================================
        # EVALUATION
        # ====================================================

        eval_strategy="epoch",

        # ====================================================
        # BEST MODEL
        # ====================================================

        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,

        # ====================================================
        # CHECKPOINT
        # ====================================================

        # Hanya disimpan sementara di storage lokal Colab.
        save_strategy="epoch",

        # ====================================================
        # LOGGING
        # ====================================================

        logging_strategy="epoch",

        # ====================================================
        # GPU / EFFICIENCY
        # ====================================================

        fp16=torch.cuda.is_available(),

        # ====================================================
        # REPRODUCIBILITY
        # ====================================================

        seed=SEED,

        # ====================================================
        # REPORTING
        # ====================================================

        report_to="none",

        # ====================================================
        # DATALOADER
        # ====================================================

        dataloader_pin_memory=torch.cuda.is_available(),
    )

    return training_args


# ============================================================
# EARLY STOPPING CONFIGURATION
# ============================================================

EARLY_STOPPING_PATIENCE = 2

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=EARLY_STOPPING_PATIENCE
)


print("=" * 60)
print("TRAINING CONFIGURATION READY")
print("=" * 60)

print("Maximum epochs      : 10")
print("Early stopping      : ENABLED")
print(f"Patience            : {EARLY_STOPPING_PATIENCE}")
print("Evaluation          : every epoch")
print("Best metric         : f1_macro")
print("Learning rate       : 2e-5")
print("Weight decay        : 0.01")
print("FP16                :", torch.cuda.is_available())
print("Checkpoint          : temporary local Colab storage")
print("Google Drive model  : NOT SAVED")
print("Status              : READY")

TRAINING CONFIGURATION READY
Maximum epochs      : 10
Early stopping      : ENABLED
Patience            : 2
Evaluation          : every epoch
Best metric         : f1_macro
Learning rate       : 2e-5
Weight decay        : 0.01
FP16                : True
Checkpoint          : temporary local Colab storage
Google Drive model  : NOT SAVED
Status              : READY


In [ ]:
# ============================================================
# CELL 19 — PREPARE ONE EXPERIMENT
# ============================================================

def prepare_experiment(
    model_label,
    max_length
):
    """
    Menyiapkan satu eksperimen fine-tuning.

    Pipeline:
    tokenizer
    → tokenisasi
    → data collator
    → model
    → TrainingArguments
    → Trainer + Early Stopping

    Training belum dijalankan di fungsi ini.
    """

    # ========================================================
    # GET MODEL CONFIGURATION
    # ========================================================

    if model_label not in MODEL_CONFIGS:
        raise ValueError(
            f"Model tidak ditemukan: {model_label}"
        )

    if max_length not in MAX_LENGTH_CANDIDATES:
        raise ValueError(
            f"MAX_LENGTH tidak valid: {max_length}. "
            f"Gunakan salah satu: {MAX_LENGTH_CANDIDATES}"
        )

    config = MODEL_CONFIGS[model_label]

    model_name = config["model_name"]
    batch_size = config["batch_size"]

    print("=" * 60)
    print("PREPARING EXPERIMENT")
    print("=" * 60)

    print(f"Model      : {model_label}")
    print(f"Checkpoint : {model_name}")
    print(f"MAX_LENGTH : {max_length}")
    print(f"Batch size : {batch_size}")

    # ========================================================
    # 1. LOAD TOKENIZER
    # ========================================================

    print("\n[1/5] Loading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True
    )

    print(
        f"Tokenizer  : {type(tokenizer).__name__}"
    )

    # ========================================================
    # 2. TOKENIZATION
    # ========================================================

    print("\n[2/5] Tokenizing datasets...")

    (
        tokenized_train,
        tokenized_validation,
        tokenized_test
    ) = tokenize_datasets(
        train_hf,
        val_hf,
        test_hf,
        tokenizer,
        max_length
    )

    # ========================================================
    # 3. DATA COLLATOR
    # ========================================================

    print("\n[3/5] Creating data collator...")

    data_collator = create_data_collator(
        tokenizer
    )

    # ========================================================
    # 4. LOAD MODEL
    # ========================================================

    print("\n[4/5] Loading model...")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id
    )

    print(
        f"Model loaded : {model.__class__.__name__}"
    )

    # ========================================================
    # 5. TRAINING ARGUMENTS
    # ========================================================

    print("\n[5/5] Creating TrainingArguments...")

    experiment_name = (
        f"{model_label.replace(' ', '_')}_MAX{max_length}"
    )

    training_args = create_training_arguments(
        experiment_name=experiment_name,
        batch_size=batch_size
    )

    # ========================================================
    # NEW EARLY STOPPING CALLBACK
    # ========================================================

    experiment_early_stopping = EarlyStoppingCallback(
        early_stopping_patience=EARLY_STOPPING_PATIENCE
    )

    # ========================================================
    # CREATE TRAINER
    # ========================================================

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_validation,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[experiment_early_stopping]
    )

    print("\n" + "=" * 60)
    print("EXPERIMENT READY")
    print("=" * 60)

    return {
        "model": model,
        "tokenizer": tokenizer,
        "tokenized_train": tokenized_train,
        "tokenized_validation": tokenized_validation,
        "tokenized_test": tokenized_test,
        "data_collator": data_collator,
        "training_args": training_args,
        "trainer": trainer,
        "model_label": model_label,
        "max_length": max_length,
        "experiment_name": experiment_name,
    }

In [ ]:
# ============================================================
# CELL 20 — RUN ONE EXPERIMENT
# ============================================================

def run_experiment(
    model_label,
    max_length
):
    """
    Menjalankan satu eksperimen:
    prepare → train → validation → save metrics → cleanup GPU.
    """

    experiment_name = (
        f"{model_label.replace(' ', '_')}_MAX{max_length}"
    )

    print("\n" + "=" * 70)
    print("START EXPERIMENT")
    print("=" * 70)
    print(f"Experiment : {experiment_name}")
    print(f"Model      : {model_label}")
    print(f"MAX_LENGTH : {max_length}")

    start_time = time.time()

    experiment = None
    trainer = None
    model = None

    try:
        # ====================================================
        # 1. PREPARE
        # ====================================================

        experiment = prepare_experiment(
            model_label=model_label,
            max_length=max_length
        )

        trainer = experiment["trainer"]
        model = experiment["model"]

        # ====================================================
        # 2. TRAIN
        # ====================================================

        print("\n" + "=" * 60)
        print("TRAINING STARTED")
        print("=" * 60)

        train_result = trainer.train()

        # ====================================================
        # 3. VALIDATION
        # ====================================================

        print("\n" + "=" * 60)
        print("VALIDATION")
        print("=" * 60)

        eval_results = trainer.evaluate(
            eval_dataset=experiment["tokenized_validation"]
        )

        # ====================================================
        # 4. BEST CHECKPOINT / EPOCH
        # ====================================================

        best_checkpoint = getattr(
            trainer.state,
            "best_model_checkpoint",
            None
        )

        best_epoch = None

        if best_checkpoint is not None:
            checkpoint_name = os.path.basename(
                best_checkpoint
            )

            if checkpoint_name.startswith("checkpoint-"):
                try:
                    best_step = int(
                        checkpoint_name.split("-")[-1]
                    )

                    best_epoch = (
                        best_step /
                        trainer.state.max_steps
                        * trainer.state.num_train_epochs
                    )

                except Exception:
                    best_epoch = None

        # Jika Trainer mencatat epoch terbaik secara langsung
        if best_epoch is None:
            if hasattr(trainer.state, "best_model_checkpoint"):
                if trainer.state.best_model_checkpoint:
                    # Cari epoch dari log history yang memiliki
                    # metric evaluasi terbaik
                    eval_logs = [
                        log for log in trainer.state.log_history
                        if "eval_f1_macro" in log
                    ]

                    if len(eval_logs) > 0:
                        best_log = max(
                            eval_logs,
                            key=lambda x: x["eval_f1_macro"]
                        )

                        best_epoch = best_log.get("epoch")

        # ====================================================
        # 5. TRAINING TIME
        # ====================================================

        training_time = time.time() - start_time

        # ====================================================
        # 6. EXTRACT METRICS
        # ====================================================

        result_row = {
            "experiment_id": experiment_name,
            "model": model_label,
            "checkpoint": MODEL_CONFIGS[model_label]["model_name"],
            "max_length": max_length,
            "batch_size": MODEL_CONFIGS[model_label]["batch_size"],
            "status": "completed",
            "best_epoch": best_epoch,
            "eval_loss": eval_results.get("eval_loss"),
            "accuracy": eval_results.get("eval_accuracy"),
            "precision": eval_results.get("eval_precision"),
            "recall": eval_results.get("eval_recall"),
            "f1_macro": eval_results.get("eval_f1_macro"),
            "training_time_seconds": training_time,
            "error": ""
        }

        # ====================================================
        # 7. SAVE RESULT IMMEDIATELY
        # ====================================================

        result_df = pd.DataFrame(
            [result_row],
            columns=RESULT_COLUMNS
        )

        result_df.to_csv(
            RESULTS_PATH,
            mode="a",
            header=False,
            index=False
        )

        print("\n" + "=" * 60)
        print("EXPERIMENT COMPLETED")
        print("=" * 60)

        print(f"Model       : {model_label}")
        print(f"MAX_LENGTH  : {max_length}")
        print(f"Best epoch  : {best_epoch}")
        print(
            f"Accuracy    : "
            f"{result_row['accuracy']:.4f}"
        )
        print(
            f"Precision   : "
            f"{result_row['precision']:.4f}"
        )
        print(
            f"Recall      : "
            f"{result_row['recall']:.4f}"
        )
        print(
            f"Macro-F1    : "
            f"{result_row['f1_macro']:.4f}"
        )
        print(
            f"Time        : "
            f"{training_time / 60:.2f} minutes"
        )

        print(f"\nResult saved to:")
        print(RESULTS_PATH)

        return result_row

    except Exception as e:

        training_time = time.time() - start_time

        error_message = repr(e)

        print("\n" + "=" * 60)
        print("EXPERIMENT FAILED")
        print("=" * 60)

        print(f"Model      : {model_label}")
        print(f"MAX_LENGTH : {max_length}")
        print(f"Error      : {error_message}")

        # Simpan status FAILED supaya eksperimen
        # tidak dianggap belum pernah dijalankan.
        failed_row = {
            "experiment_id": experiment_name,
            "model": model_label,
            "checkpoint": MODEL_CONFIGS[model_label]["model_name"],
            "max_length": max_length,
            "batch_size": MODEL_CONFIGS[model_label]["batch_size"],
            "status": "failed",
            "best_epoch": None,
            "eval_loss": None,
            "accuracy": None,
            "precision": None,
            "recall": None,
            "f1_macro": None,
            "training_time_seconds": training_time,
            "error": error_message
        }

        pd.DataFrame(
            [failed_row],
            columns=RESULT_COLUMNS
        ).to_csv(
            RESULTS_PATH,
            mode="a",
            header=False,
            index=False
        )

        return failed_row

    finally:

        # ====================================================
        # 8. GPU / MEMORY CLEANUP
        # ====================================================

        print("\nCleaning experiment resources...")

        if trainer is not None:
            del trainer

        if model is not None:
            del model

        if experiment is not None:
            del experiment

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        print("GPU cleanup completed.")

In [ ]:
# ============================================================
# CELL 21 — TRAINING PIPELINE SMOKE TEST
# ============================================================

TEST_MODEL = "IndoBERT Base P1"
TEST_MAX_LENGTH = 256

print("=" * 70)
print("TRAINING PIPELINE SMOKE TEST")
print("=" * 70)

print(f"Model      : {TEST_MODEL}")
print(f"MAX_LENGTH : {TEST_MAX_LENGTH}")
print()
print("Tujuan:")
print("- Memastikan pipeline training berjalan")
print("- Memastikan Early Stopping berjalan")
print("- Memastikan GPU dan Trainer aman")
print()
print("CATATAN:")
print("Hasil ini BUKAN bagian dari 18 eksperimen penelitian.")
print("=" * 70)

# Siapkan eksperimen
experiment = prepare_experiment(
    model_label=TEST_MODEL,
    max_length=TEST_MAX_LENGTH
)

trainer = experiment["trainer"]

# Training
print("\n" + "=" * 70)
print("SMOKE TEST TRAINING STARTED")
print("=" * 70)

smoke_train_result = trainer.train()

# Validation
print("\n" + "=" * 70)
print("SMOKE TEST VALIDATION")
print("=" * 70)

smoke_eval = trainer.evaluate(
    eval_dataset=experiment["tokenized_validation"]
)

print("\n" + "=" * 70)
print("SMOKE TEST FINISHED")
print("=" * 70)

print("\nValidation results:")
for key, value in smoke_eval.items():
    print(f"{key}: {value}")

# ============================================================
# CLEANUP
# ============================================================

del trainer
del experiment

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("\nGPU cleanup completed.")
print("Smoke test selesai.")
print("Hasil smoke test TIDAK disimpan ke results.csv.")

TRAINING PIPELINE SMOKE TEST
Model      : IndoBERT Base P1
MAX_LENGTH : 256

Tujuan:
- Memastikan pipeline training berjalan
- Memastikan Early Stopping berjalan
- Memastikan GPU dan Trainer aman

CATATAN:
Hasil ini BUKAN bagian dari 18 eksperimen penelitian.
PREPARING EXPERIMENT
Model      : IndoBERT Base P1
Checkpoint : indobenchmark/indobert-base-p1
MAX_LENGTH : 256
Batch size : 16

[1/5] Loading tokenizer...
Tokenizer  : BertTokenizer

[2/5] Tokenizing datasets...


Tokenizing train | max_length=256:   0%|          | 0/3052 [00:00<?, ? examples/s]

Tokenizing validation | max_length=256:   0%|          | 0/381 [00:00<?, ? examples/s]

Tokenizing test | max_length=256:   0%|          | 0/382 [00:00<?, ? examples/s]


[3/5] Creating data collator...

[4/5] Loading model...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


SMOKE TEST VALIDATION


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Macro
0.190311,1.334194,5,0.503937,0.510758,0.504170,0.499605



SMOKE TEST FINISHED

Validation results:
eval_loss: 1.3341941833496094
eval_accuracy: 0.5039370078740157
eval_precision: 0.5107584142496492
eval_recall: 0.5041695146958305
eval_f1_macro: 0.49960485967879265

GPU cleanup completed.
Smoke test selesai.
Hasil smoke test TIDAK disimpan ke results.csv.


In [ ]:
# ============================================================
# CELL 22 — CHECK EXPERIMENT RESULTS
# ============================================================

results_df = pd.read_csv(RESULTS_PATH)

print("=" * 60)
print("EXPERIMENT RESULTS CHECK")
print("=" * 60)

print(f"Results file : {RESULTS_PATH}")
print(f"Recorded experiments : {len(results_df)}")

if len(results_df) > 0:
    display(results_df)
else:
    print("Tidak ada hasil eksperimen.")
    print("Ready untuk 18 eksperimen final.")

EXPERIMENT RESULTS CHECK
Results file : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/results/indobert_18_experiments_results.csv
Recorded experiments : 0
Tidak ada hasil eksperimen.
Ready untuk 18 eksperimen final.


In [ ]:
# ============================================================
# CELL 23 — RUN 18 FINAL EXPERIMENTS
# 6 MODELS × 3 MAX_LENGTH
# ============================================================

print("=" * 70)
print("FINAL EXPERIMENTS")
print("=" * 70)

print("Models      :", len(MODEL_CONFIGS))
print("MAX_LENGTH  :", MAX_LENGTH_CANDIDATES)
print(
    "Total       :",
    len(MODEL_CONFIGS) * len(MAX_LENGTH_CANDIDATES)
)

# ============================================================
# LOAD EXISTING RESULTS
# ============================================================

results_df = pd.read_csv(RESULTS_PATH)

completed_experiments = set(
    results_df.loc[
        results_df["status"] == "completed",
        "experiment_id"
    ].astype(str)
)

print(
    f"\nCompleted experiments already recorded : "
    f"{len(completed_experiments)}"
)

if len(completed_experiments) > 0:
    print("\nExperiments yang akan di-skip:")
    for exp in sorted(completed_experiments):
        print(f"- {exp}")


# ============================================================
# BUILD EXPERIMENT LIST
# ============================================================

experiment_list = []

experiment_number = 1

for model_label in MODEL_CONFIGS:

    for max_length in MAX_LENGTH_CANDIDATES:

        experiment_name = (
            f"{model_label.replace(' ', '_')}_MAX{max_length}"
        )

        experiment_list.append({
            "number": experiment_number,
            "experiment_id": experiment_name,
            "model_label": model_label,
            "max_length": max_length
        })

        experiment_number += 1


# ============================================================
# RUN EXPERIMENTS
# ============================================================

for exp in experiment_list:

    experiment_number = exp["number"]
    experiment_id = exp["experiment_id"]
    model_label = exp["model_label"]
    max_length = exp["max_length"]

    # --------------------------------------------------------
    # SKIP COMPLETED EXPERIMENT
    # --------------------------------------------------------

    if experiment_id in completed_experiments:

        print("\n" + "=" * 70)
        print(
            f"SKIP EXPERIMENT "
            f"{experiment_number}/18"
        )
        print("=" * 70)

        print(
            f"{experiment_id} "
            f"already completed."
        )

        continue

    # --------------------------------------------------------
    # RUN
    # --------------------------------------------------------

    print("\n\n" + "#" * 70)
    print(
        f"EXPERIMENT "
        f"{experiment_number}/18"
    )
    print("#" * 70)

    print(f"Model      : {model_label}")
    print(f"MAX_LENGTH : {max_length}")

    result = run_experiment(
        model_label=model_label,
        max_length=max_length
    )

    # --------------------------------------------------------
    # UPDATE COMPLETED SET
    # --------------------------------------------------------

    if result["status"] == "completed":

        completed_experiments.add(
            experiment_id
        )

    # --------------------------------------------------------
    # SHOW PROGRESS
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print(
        f"PROGRESS: "
        f"{len(completed_experiments)}/18 completed"
    )
    print("-" * 70)


# ============================================================
# FINAL SUMMARY
# ============================================================

results_df = pd.read_csv(
    RESULTS_PATH
)

print("\n" + "=" * 70)
print("FINAL EXPERIMENT SUMMARY")
print("=" * 70)

print(
    f"Total recorded : "
    f"{len(results_df)}"
)

print(
    f"Completed       : "
    f"{(results_df['status'] == 'completed').sum()}"
)

print(
    f"Failed          : "
    f"{(results_df['status'] == 'failed').sum()}"
)

display(results_df)

FINAL EXPERIMENTS
Models      : 6
MAX_LENGTH  : [128, 256, 512]
Total       : 18

Completed experiments already recorded : 0


######################################################################
EXPERIMENT 1/18
######################################################################
Model      : IndoBERT Lite P1
MAX_LENGTH : 128

START EXPERIMENT
Experiment : IndoBERT_Lite_P1_MAX128
Model      : IndoBERT Lite P1
MAX_LENGTH : 128
PREPARING EXPERIMENT
Model      : IndoBERT Lite P1
Checkpoint : indobenchmark/indobert-lite-base-p1
MAX_LENGTH : 128
Batch size : 16

[1/5] Loading tokenizer...


config.json:   0%|          | 0.00/1.54k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer  : AlbertTokenizer

[2/5] Tokenizing datasets...


...


[3/5] Creating data collator...

[4/5] Loading model...


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.34GB            

pytorch_model.bin: downloading bytes:           |  0.00B            


EXPERIMENT FAILED
Model      : IndoBERT Large P2
MAX_LENGTH : 512
Error      : OSError("Can't load the model for 'indobenchmark/indobert-large-p2'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'indobenchmark/indobert-large-p2' is the correct path to a directory containing a file named pytorch_model.bin.")

Cleaning experiment resources...
GPU cleanup completed.

----------------------------------------------------------------------
PROGRESS: 13/18 completed
----------------------------------------------------------------------

FINAL EXPERIMENT SUMMARY
Total recorded : 18
Completed       : 13
Failed          : 5


,experiment_id,model,checkpoint,max_length,batch_size,status,best_epoch,eval_loss,accuracy,precision,recall,f1_macro,training_time_seconds,error
0,IndoBERT_Lite_P1_MAX128,IndoBERT Lite P1,indobenchmark/indobert-lite-base-p1,128,16,completed,5.0,1.537792,0.270341,0.322164,0.270711,0.245238,91.421959,NaN
1,IndoBERT_Lite_P1_MAX256,IndoBERT Lite P1,indobenchmark/indobert-lite-base-p1,256,16,completed,1.0,1.578397,0.244094,0.275853,0.244532,0.220760,46.701091,NaN
2,IndoBERT_Lite_P1_MAX512,IndoBERT Lite P1,indobenchmark/indobert-lite-base-p1,512,16,completed,3.0,1.542539,0.309711,0.265669,0.309159,0.259564,82.544849,NaN
3,IndoBERT_Lite_P2_MAX128,IndoBERT Lite P2,indobenchmark/indobert-lite-base-p2,128,16,completed,5.0,1.538006,0.296588,0.237580,0.296890,0.258244,93.236050,NaN
4,IndoBERT_Lite_P2_MAX256,IndoBERT Lite P2,indobenchmark/indobert-lite-base-p2,256,16,completed,6.0,1.551229,0.296588,0.329775,0.297129,0.259322,119.480417,NaN
5,IndoBERT_Lite_P2_MAX512,IndoBERT Lite P2,indobenchmark/indobert-lite-base-p2,512,16,completed,2.0,1.553601,0.265092,0.259675,0.265789,0.226610,67.416425,NaN
6,IndoBERT_Base_P1_MAX128,IndoBERT Base P1,indobenchmark/indobert-base-p1,128,16,completed,3.0,1.378493,0.493438,0.501060,0.493609,0.491131,403.371578,NaN
7,IndoBERT_Base_P1_MAX256,IndoBERT Base P1,indobenchmark/indobert-base-p1,256,16,completed,3.0,1.340257,0.485564,0.483709,0.485885,0.480689,331.333117,NaN
8,IndoBERT_Base_P1_MAX512,IndoBERT Base P1,indobenchmark/indobert-base-p1,512,16,completed,3.0,1.335024,0.501312,0.498173,0.501606,0.498900,335.865187,NaN
9,IndoBERT_Base_P2_MAX128,IndoBERT Base P2,indobenchmark/indobert-base-p2,128,16,completed,3.0,1.372166,0.485564,0.485261,0.485783,0.485163,297.657983,NaN
